# Econometría II — Tarea 1
**Mínimos Cuadrados Ordinarios, Predicción y Evaluación de Políticas**

Datos NLS. Modelo base: $lwage = \beta_0 + \beta_1\,educ + \beta_2\,exper + \beta_3\,exper^2 + u$, con $exper = age - educ - 6$ (experiencia potencial).

In [4]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Cargar datos
datos = pd.read_stata('NLS80V2.dta')   # ajustar la ruta si el archivo está en otra carpeta
datos[['lwage', 'educ', 'exper', 'age']].describe().round(3)

,lwage,educ,exper,age
count,935.000,935.000,935.000,935.000
mean,6.779,13.468,13.612,33.080
std,0.421,2.197,3.828,3.108
min,4.745,9.000,5.000,28.000
25%,6.506,12.000,11.000,30.000
50%,6.808,12.000,13.000,33.000
75%,7.056,16.000,17.000,36.000
max,8.032,18.000,23.000,38.000


## Pregunta 1 — Estimación del modelo por MCO

Se reportan coeficientes y errores estándar. El término $exper^2$ permite un perfil salario-experiencia cóncavo ($\hat\beta_3<0$).

In [5]:
datos['exper2'] = datos['exper']**2
X = sm.add_constant(datos[['educ', 'exper', 'exper2']])
modelo = sm.OLS(datos['lwage'], X).fit()

tabla1 = pd.DataFrame({'coeficiente': modelo.params, 'error estándar': modelo.bse})
print(tabla1.round(5))
print(f"\nn = {int(modelo.nobs)},  R² = {modelo.rsquared:.4f}")

        coeficiente  error estándar
const       5.05292         0.21333
educ        0.08383         0.00725
exper       0.06710         0.02394
exper2     -0.00158         0.00084

n = 935,  R² = 0.1282


In [6]:
# Coeficientes que se usan en todo lo que sigue
b1 = modelo.params['educ']
b2 = modelo.params['exper']
b3 = modelo.params['exper2']

## Pregunta 2 — Efecto de aumentar en 1 año la educación de todos

Como $exper = age - educ - 6$, la política mueve **dos** variables: $educ \to educ+1$ y $exper \to exper-1$ (a edad fija). El efecto individual (predicción después $-$ antes) es:

$$\Delta_i = \beta_1 - \beta_2 + \beta_3(1 - 2\,exper_i)$$

El efecto sobre el $lwage$ **promedio** es el promedio de los $\Delta_i$; como $\Delta_i$ es lineal en $exper_i$, equivale a evaluar en $\bar e$:

$$\bar\Delta = \beta_1 - \beta_2 + \beta_3(1 - 2\bar e)$$

In [7]:
e_barra = datos['exper'].mean()
efecto2 = b1 - b2 + b3 * (1 - 2*e_barra)
print(f"exper promedio (observada, pre-política): {e_barra:.4f}")
print(f"Efecto sobre lwage promedio: {efecto2:.5f}  (~{100*efecto2:.2f}% en el salario)")

exper promedio (observada, pre-política): 13.6118
Efecto sobre lwage promedio: 0.05822  (~5.82% en el salario)


In [8]:
# Verificación por predicción: dos mundos explícitos
X_pol2 = sm.add_constant(pd.DataFrame({
    'educ':   datos['educ'] + 1,
    'exper':  datos['exper'] - 1,
    'exper2': (datos['exper'] - 1)**2
}))
verif2 = (modelo.predict(X_pol2) - modelo.predict(X)).mean()
print(f"Verificación por predicción: {verif2:.5f}   (debe coincidir con {efecto2:.5f})")

Verificación por predicción: 0.05822   (debe coincidir con 0.05822)


## Pregunta 3 — El efecto como coeficiente de una regresión (reparametrización)

Sea $\theta \equiv \beta_1 - \beta_2 - \beta_3(2\bar e - 1)$. Receta *despeja–sustituye–reagrupa*: despejando $\beta_1 = \theta + \beta_2 + \beta_3(2\bar e - 1)$ y reagrupando en el modelo original:

$$lwage = \beta_0 + \theta\,educ + \beta_2\,\underbrace{(educ + exper)}_{z_1} + \beta_3\,\underbrace{(exper^2 + (2\bar e - 1)\,educ)}_{z_2} + u$$

Es el **mismo modelo en otras coordenadas** (mismo $R^2$, mismos residuos): el coeficiente de $educ$ pasa a ser $\theta$, con su error estándar MCO directo en la tabla.

In [ ]:
a = 1 - 2*e_barra
datos['z1'] = datos['educ'] + datos['exper']
datos['z2'] = datos['exper2'] - a * datos['educ']   # = exper2 + (2ē-1)·educ

X3 = sm.add_constant(datos[['educ', 'z1', 'z2']])
modelo3 = sm.OLS(datos['lwage'], X3).fit()

print(f"θ (coef. de educ):    {modelo3.params['educ']:.5f}   (= efecto P2: {efecto2:.5f})")
print(f"SE(θ):                {modelo3.bse['educ']:.5f}")
print(f"t = {modelo3.tvalues['educ']:.3f}   ->  H0: 'la política no tiene efecto'")

In [ ]:
# Chequeos: misma regresión reparametrizada + verificación con t_test
print("z1 recupera β2:", round(modelo3.params['z1'], 5), "=", round(b2, 5))
print("z2 recupera β3:", round(modelo3.params['z2'], 5), "=", round(b3, 5))
print("Mismo R²:", round(modelo.rsquared, 6), "=", round(modelo3.rsquared, 6))

# Verificación cruzada: combinación lineal sobre el modelo original
tt2 = modelo.t_test(f"educ - exper - {2*e_barra - 1}*exper2 = 0")
print("\nt_test (modelo original):", tt2.effect.item().round(5), "SE:", np.sqrt(tt2.var).item().round(5))

## Pregunta 4 — Política: educación mínima de 12 años

Aumento individual: $d_i = \max(12 - educ_i,\, 0)$ (heterogéneo; $d_i = 0$ para los no afectados). Por la mecánica de la experiencia potencial: $educ_i \to educ_i + d_i$, $exper_i \to exper_i - d_i$. El efecto individual:

$$\Delta_i = \beta_1 d_i - \beta_2 d_i + \beta_3\,(d_i^2 - 2\,exper_i\, d_i)$$

Con $d_i=1$ se recupera la fórmula de la P2; con $d_i=0$, $\Delta_i=0$. El promedio se toma sobre **toda** la muestra (los ceros diluyen). No hay atajo de medias: $d_i$ y $exper_i$ aparecen multiplicados, así que se promedia la columna de efectos individuales.

In [ ]:
d = (12 - datos['educ']).clip(lower=0)

delta_i = b1*d - b2*d + b3*(d**2 - 2*datos['exper']*d)
efecto4 = delta_i.mean()

print(f"Personas afectadas: {(d > 0).sum()} de {len(datos)}")
print(f"Aumento promedio de educación (d̄): {d.mean():.4f} años")
print(f"Efecto sobre lwage promedio: {efecto4:.5f}  (~{100*efecto4:.2f}% en el salario)")

In [ ]:
# Verificación por predicción
educ_pol  = datos['educ'].clip(lower=12)
exper_pol = datos['exper'] - d
X_pol4 = sm.add_constant(pd.DataFrame({
    'educ': educ_pol, 'exper': exper_pol, 'exper2': exper_pol**2
}))
verif4 = (modelo.predict(X_pol4) - modelo.predict(X)).mean()
print(f"Verificación por predicción: {verif4:.5f}   (debe coincidir con {efecto4:.5f})")

*Nota (nivel vs. logaritmo):* el enunciado dice "nivel promedio de ingresos". El efecto reportado está en unidades de $lwage$ (cambio proporcional aproximado, $\times 100 \approx \%$). La traducción a niveles monetarios ($wage_i \cdot e^{\Delta_i}$, con residuo fijo) se muestra como complemento; su inferencia sería no lineal en $\hat\beta$ y queda fuera de la P5.

In [ ]:
# Complemento: traducción a niveles (supuesto: residuo individual invariante a la política)
if 'wage' in datos.columns:
    wage_obs = datos['wage']
else:
    wage_obs = np.exp(datos['lwage'])
efecto4_niveles = (wage_obs * np.exp(delta_i) - wage_obs).mean()
print(f"Aumento promedio del salario en niveles: {efecto4_niveles:.2f} (unidades de wage)")

## Pregunta 5 — Error estándar del efecto de la P4

El efecto es una **combinación lineal** de los coeficientes: agrupando el promedio por coeficiente,

$$\bar\Delta = \beta_1 w_1 + \beta_2 w_2 + \beta_3 w_3 = c'\beta, \qquad c = (0,\; \bar d,\; -\bar d,\; \overline{d^2 - 2\,exper\,d})'$$

Los pesos se tratan como fijos (análisis condicional en $X$); la única aleatoriedad es $\hat\beta$. La varianza de una combinación lineal requiere varianzas **y covarianzas**:

$$Var(c'\hat\beta) = c'\,\hat V\,c \;=\; \sum_j\sum_k c_j c_k \hat V_{jk} \qquad\Rightarrow\qquad SE = \sqrt{c'\hat V c}$$

In [ ]:
# Vector de pesos c (orden: const, educ, exper, exper2)
w1 = d.mean()
w2 = -d.mean()
w3 = (d**2 - 2*datos['exper']*d).mean()
c  = np.array([0, w1, w2, w3])

print("c =", c.round(4))
print("Chequeo c'β̂ = efecto P4:", (c @ modelo.params.values).round(5), "=", efecto4.round(5))

### 5a. Errores estándar clásicos ($\hat V = \hat\sigma^2 (X'X)^{-1}$)

In [ ]:
V_clasica = modelo.cov_params().values
se_clasico = np.sqrt(c @ V_clasica @ c)
gl = int(modelo.df_resid)
from scipy import stats
tcrit = stats.t.ppf(0.975, gl)

print(f"SE clásico:  {se_clasico:.5f}")
print(f"t = {efecto4/se_clasico:.3f}  (gl = {gl})")
print(f"IC 95%: [{efecto4 - tcrit*se_clasico:.5f}, {efecto4 + tcrit*se_clasico:.5f}]")

# Verificación con t_test (misma cuenta hecha por statsmodels)
print("\n", modelo.t_test(c))

### 5b. Construcción manual de $\hat V$ clásica (desde los residuos)

$\hat\sigma^2 = \dfrac{\sum_i \hat u_i^2}{n-k}$ con $\hat u_i = y_i - \hat y_i$; luego $\hat V = \hat\sigma^2 (X'X)^{-1}$.

In [ ]:
u  = modelo.resid.values
Xm = X.values
n, k = Xm.shape

sigma2   = (u**2).sum() / (n - k)
V_manual = sigma2 * np.linalg.inv(Xm.T @ Xm)

print(f"σ̂² = {sigma2:.5f}")
print("V manual == cov_params:", np.allclose(V_manual, V_clasica))
print(f"SE manual del efecto: {np.sqrt(c @ V_manual @ c):.5f}")

### 5c. Errores estándar robustos HC1 (sándwich de White)

$$\hat V_{HC1} = \tfrac{n}{n-k}\,(X'X)^{-1}\Big(\textstyle\sum_i \hat u_i^2\, x_i x_i'\Big)(X'X)^{-1}$$

Los $\hat\beta$ (y el efecto puntual) **no cambian**; solo cambia la matriz que entra a $c'\hat V c$. Bajo homocedasticidad la "carne" colapsa: $\sum \hat u_i^2 x_i x_i' \approx \hat\sigma^2 X'X$ y el sándwich se reduce a la clásica.

In [ ]:
# Construcción manual del sándwich
carne  = Xm.T @ np.diag(u**2) @ Xm          # Σ û_i² · x_i x_i'
pan    = np.linalg.inv(Xm.T @ Xm)
V_hc1m = (n/(n-k)) * pan @ carne @ pan

# Verificación contra statsmodels
modelo_r = sm.OLS(datos['lwage'], X).fit(cov_type='HC1')
print("V_HC1 manual == statsmodels:", np.allclose(V_hc1m, modelo_r.cov_params().values))

se_hc1 = np.sqrt(c @ V_hc1m @ c)
print(f"\nEfecto (idéntico):  {(c @ modelo_r.params.values):.5f}")
print(f"SE clásico:  {se_clasico:.5f}")
print(f"SE HC1:      {se_hc1:.5f}")
print(f"IC 95% (HC1): [{efecto4 - tcrit*se_hc1:.5f}, {efecto4 + tcrit*se_hc1:.5f}]")

### 5d. Verificación por reparametrización (misma receta de la P3)

Despejando $\beta_1$ de $\theta_4 = c'\beta$ y reagrupando, la regresión de $lwage$ sobre $\big(educ/\bar d,\; educ+exper,\; exper^2 - (w_3/\bar d)\,educ\big)$ entrega $\theta_4$ como coeficiente, con su SE directo en la tabla. Debe coincidir al decimal con 5a (y con 5c si se estima con `HC1`).

In [ ]:
datos['x_star'] = datos['educ'] / d.mean()
datos['z2p']    = datos['exper2'] - (w3 / d.mean()) * datos['educ']

X5 = sm.add_constant(datos[['x_star', 'z1', 'z2p']])

m5_cl = sm.OLS(datos['lwage'], X5).fit()
m5_r  = sm.OLS(datos['lwage'], X5).fit(cov_type='HC1')

print(f"θ4 (coef x*):      {m5_cl.params['x_star']:.5f}   (= {efecto4:.5f})")
print(f"SE clásico:        {m5_cl.bse['x_star']:.5f}   (= {se_clasico:.5f})")
print(f"SE HC1:            {m5_r.bse['x_star']:.5f}   (= {se_hc1:.5f})")
print(f"Mismo R² que el modelo original: {np.isclose(m5_cl.rsquared, modelo.rsquared)}")

## Resumen de resultados

| Pregunta | Objeto | Resultado |
|---|---|---|
| 1 | $\hat\beta$, SE | tabla MCO |
| 2 | $\bar\Delta_{+1} = \beta_1-\beta_2+\beta_3(1-2\bar e)$ | `efecto2` |
| 3 | $\theta$ como coef. de regresión reparametrizada | `modelo3` |
| 4 | $\bar\Delta_{12} = c'\hat\beta$ | `efecto4` |
| 5 | $SE = \sqrt{c'\hat V c}$ (clásico y HC1, 3 vías coincidentes) | `se_clasico`, `se_hc1` |

**Interpretación P4–P5:** la política de llevar a 12 años la educación mínima aumenta el salario promedio en $\approx 100\cdot\bar\Delta\,\%$; con $t = \bar\Delta / SE$ se contrasta $H_0$: *la política no tiene efecto sobre el salario promedio* (que no es lo mismo que $\beta_1 = 0$).